# Model 

## 

In [ ]:
!uv pip install pandas
!uv pip install scikit-learn 
!uv pip install joblib

In [1]:
#### 1

import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report
import joblib

# 1. Load data
df = pd.read_csv('../data/processed/metricas_centralidade.csv')
print(f'Total stops: {len(df):,}')

# 2. Create labels
q66 = df['betweenness_centrality'].quantile(0.66)
q90 = df['betweenness_centrality'].quantile(0.90)

def classify(bc):
    if bc >= q90: return 2      # hub
    elif bc >= q66: return 1    # intermediate
    else: return 0              # peripheral

df['label'] = df['betweenness_centrality'].apply(classify)
print(f'\nClasses:')
for label, name in {0: 'Peripheral', 1: 'Intermediate', 2: 'Hub'}.items():
    print(f'  {name}: {(df.label == label).sum():,}')

# 3. Features & split
features = ['degree', 'degree_centrality', 'betweenness_centrality', 'closeness_centrality']
X = df[features].values
y = df['label'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 4. Treinar
model = GradientBoostingClassifier(n_estimators=100, max_depth=4, random_state=42)
model.fit(X_train, y_train)

# 5. Evaluate
y_pred = model.predict(X_test)
print('\n' + classification_report(y_test, y_pred, target_names=['Peripheral', 'Intermediate', 'Hub']))

scores = cross_val_score(model, X, y, cv=5, scoring='f1_macro')
print(f'F1 macro (CV): {scores.mean():.3f} +/- {scores.std():.3f}')

# 6. Save
joblib.dump(model, 'sp_transit_classifier.joblib')
print('\nModelo salvo: sp_transit_classifier.joblib')

Total stops: 21,892

Classes:
  Peripheral: 14,449
  Intermediate: 5,253
  Hub: 2,190

              precision    recall  f1-score   support

  Peripheral       1.00      1.00      1.00      2890
Intermediate       1.00      1.00      1.00      1051
         Hub       1.00      1.00      1.00       438

    accuracy                           1.00      4379
   macro avg       1.00      1.00      1.00      4379
weighted avg       1.00      1.00      1.00      4379

F1 macro (CV): 1.000 +/- 0.000

Modelo salvo: sp_transit_classifier.joblib


- F1 de 1.000: O modelo está com 100% porque as labels são derivadas diretamente do `betweenness_centrality`, que é uma das *features*. Ele tá aprendendo a regra dos quantis, não algo útil.
  

- Duas opções:
| Opção | Abordagem | Resultado | 
|---|---|---|
| A. Remover *betweenness* das *features* | Prever a categoria usando só *degree*, *degree_centrality* e *closeness* | Modelo honesto e interessante | 
| B. Mudar o target | Prever outra coisa (ex: se a parada é terminal, se tem alto grau) | Mais criativo |


Escolha: opção A: é o que faz sentido academicamente: "Consigo identificar gargalos da rede sem calcular *betweenness* (que é computacionalmente caro)?"

In [2]:
#### 2 ####


import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report
import joblib

df = pd.read_csv('../data/processed/metricas_centralidade.csv')

# Labels baseadas em betweenness
q66 = df['betweenness_centrality'].quantile(0.66)
q90 = df['betweenness_centrality'].quantile(0.90)

def classify(bc):
    if bc >= q90: return 2
    elif bc >= q66: return 1
    else: return 0

df['label'] = df['betweenness_centrality'].apply(classify)

# Features SEM betweenness (o target é derivado dele)
features = ['degree', 'degree_centrality', 'closeness_centrality']
X = df[features].values
y = df['label'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = GradientBoostingClassifier(n_estimators=100, max_depth=4, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=['Peripheral', 'Intermediate', 'Hub']))

scores = cross_val_score(model, X, y, cv=5, scoring='f1_macro')
print(f'F1 macro (CV): {scores.mean():.3f} +/- {scores.std():.3f}')

joblib.dump(model, 'sp_transit_classifier.joblib')
print('Model saved!')

              precision    recall  f1-score   support

  Peripheral       0.72      0.93      0.81      2890
Intermediate       0.39      0.18      0.25      1051
         Hub       0.41      0.12      0.19       438

    accuracy                           0.67      4379
   macro avg       0.51      0.41      0.42      4379
weighted avg       0.61      0.67      0.61      4379

F1 macro (CV): 0.419 +/- 0.027
Model saved!


O problema principal é o desbalanceamento de classes e poucas *features*.
Adicionar as coordenadas geográficas (posição importa na rede) e balancear as classes

In [3]:
#### 3 ####


import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report
import joblib

df = pd.read_csv('../data/processed/metricas_centralidade.csv')

# Labels
q66 = df['betweenness_centrality'].quantile(0.66)
q90 = df['betweenness_centrality'].quantile(0.90)

def classify(bc):
    if bc >= q90: return 2
    elif bc >= q66: return 1
    else: return 0

df['label'] = df['betweenness_centrality'].apply(classify)

# Features: COM betweenness, mas SEM coordenadas
features = ['degree', 'degree_centrality', 'closeness_centrality', 'lat', 'lon']
X = df[features].values
y = df['label'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Class weights to compensate imbalanced classes
from sklearn.utils.class_weight import compute_sample_weight
sample_weights = compute_sample_weight('balanced', y_train)

model = GradientBoostingClassifier(
    n_estimators=200,
    max_depth=5,
    min_samples_leaf=10,
    random_state=42,
)
model.fit(X_train, y_train, sample_weight=sample_weights)

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=['Peripheral', 'Intermediate', 'Hub']))

scores = cross_val_score(model, X, y, cv=5, scoring='f1_macro')
print(f'F1 macro (CV): {scores.mean():.3f} +/- {scores.std():.3f}')

# Feature importance
print('\nFeature Importance:')
for name, imp in sorted(zip(features, model.feature_importances_), key=lambda x: -x[1]):
    print(f'  {name:25s} {imp:.4f}')

joblib.dump(model, 'sp_transit_classifier.joblib')
print('\nModel saved!')

              precision    recall  f1-score   support

  Peripheral       0.85      0.75      0.80      2890
Intermediate       0.44      0.48      0.46      1051
         Hub       0.41      0.64      0.50       438

    accuracy                           0.68      4379
   macro avg       0.57      0.62      0.59      4379
weighted avg       0.71      0.68      0.69      4379

F1 macro (CV): 0.426 +/- 0.049

Feature Importance:
  lat                       0.2793
  lon                       0.2604
  closeness_centrality      0.2566
  degree                    0.1061
  degree_centrality         0.0976

Model saved!


F1 macro subiu de 0.42 pra 0.59 no teste, e os resultados fazem sentido: latitude e longitude são as *features* mais importantes porque *hubs* se concentram em regiões específicas de SP

### Upload model to Hugging Face Hub

In [ ]:
!uv pip install huggingface_hub

In [ ]:
from huggingface_hub import HfApi, create_repo
import json

REPO_ID = 'cintia-shinoda/sp-transit-node-classifier'

# 1. Criar repo
create_repo(repo_id=REPO_ID, exist_ok=True)
print('Repo criado')

api = HfApi()

# 2. Upload do modelo
api.upload_file(
    path_or_fileobj='sp_transit_classifier.joblib',
    path_in_repo='model.joblib',
    repo_id=REPO_ID,
)
print('Modelo uploaded')

# 3. Upload de config
config = {
    'model_type': 'GradientBoostingClassifier',
    'framework': 'scikit-learn',
    'features': ['degree', 'degree_centrality', 'closeness_centrality', 'lat', 'lon'],
    'label_map': {'0': 'Peripheral', '1': 'Intermediate', '2': 'Hub'},
    'metrics': {
        'f1_macro_test': 0.59,
        'accuracy_test': 0.68,
        'f1_macro_cv': 0.426,
    },
    'training': {
        'n_estimators': 200,
        'max_depth': 5,
        'min_samples_leaf': 10,
        'class_balancing': 'sample_weight_balanced',
        'test_size': 0.2,
    },
    'feature_importance': {
        'lat': 0.2793,
        'lon': 0.2604,
        'closeness_centrality': 0.2566,
        'degree': 0.1061,
        'degree_centrality': 0.0976,
    },
}

with open('config.json', 'w') as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

api.upload_file(
    path_or_fileobj='config.json',
    path_in_repo='config.json',
    repo_id=REPO_ID,
)
print('Config uploaded')
print(f'\nhttps://huggingface.co/{REPO_ID}')

Model Card


---
license: mit
language:
  - pt
tags:
  - tabular-classification
  - graph-theory
  - urban-mobility
  - public-transport
  - scikit-learn
  - sao-paulo
  - brazil
library_name: sklearn
datasets:
  - cintia-shinoda/sp-transit-network-centrality
metrics:
  - f1
  - accuracy
---

# SP Transit Node Classifier

Classifies bus stops in São Paulo's transit network as **Hub**, **Intermediate**, or **Peripheral** based on graph features and geographic coordinates.

The goal: **predict betweenness centrality class without computing betweenness itself** (which is computationally expensive for large networks).

## How to Use
```python
import joblib
import numpy as np
from huggingface_hub import hf_hub_download

path = hf_hub_download(
    repo_id="cintia-shinoda/sp-transit-node-classifier",
    filename="model.joblib",
)
model = joblib.load(path)

# Input: [degree, degree_centrality, closeness_centrality, lat, lon]
node = np.array([[8, 0.00036, 0.018, -23.55, -46.63]])
pred = model.predict(node)
# 0 = Peripheral, 1 = Intermediate, 2 = Hub
```

## Features

| Feature | Description |
|---------|-------------|
| degree | Number of direct connections |
| degree_centrality | Normalized degree centrality |
| closeness_centrality | Closeness centrality |
| lat | Latitude |
| lon | Longitude |

## Metrics

| Metric | Value |
|--------|-------|
| F1 Macro (test) | 0.59 |
| Accuracy (test) | 0.68 |
| F1 Macro (5-fold CV) | 0.43 |

## Feature Importance

| Feature | Importance |
|---------|-----------|
| lat | 0.2793 |
| lon | 0.2604 |
| closeness_centrality | 0.2566 |
| degree | 0.1061 |
| degree_centrality | 0.0976 |

## Key Finding

Geographic position (lat/lon) is the strongest predictor of hub status, confirming that high-centrality stops concentrate in specific corridors of São Paulo.

## Limitations

- Labels derived from betweenness centrality quantiles — simplified classification
- Trained on a single GTFS snapshot — may not generalize to network changes
- Does not consider temporal patterns (peak vs. off-peak)
- Class imbalance: 66% Peripheral, 24% Intermediate, 10% Hub

## Dataset

[SP Transit Network Centrality](https://huggingface.co/datasets/cintia-shinoda/sp-transit-network-centrality) — 21,892 bus stops with graph centrality metrics.

## Citation
```bibtex
@misc{shinoda2026sp-classifier,
  author = {Cintia Shinoda},
  title = {SP Transit Node Classifier},
  year = {2026},
  publisher = {Hugging Face},
  url = {https://huggingface.co/cintia-shinoda/sp-transit-node-classifier}
}
```

In [2]:
from huggingface_hub import HfApi
api = HfApi()
api.create_repo(
    repo_id='cintia-shinoda/sp-transit-explorer',
    repo_type='space',
    space_sdk='gradio',
    exist_ok=True,
)
print('Space criado!')

Space criado!


In [3]:
from huggingface_hub import HfApi
api = HfApi()

api.upload_file(
    path_or_fileobj='/tmp/app.py',
    path_in_repo='app.py',
    repo_id='cintia-shinoda/sp-transit-explorer',
    repo_type='space',
)
print('app.py uploaded')
print('https://huggingface.co/spaces/cintia-shinoda/sp-transit-explorer')

app.py uploaded
https://huggingface.co/spaces/cintia-shinoda/sp-transit-explorer
